<a href="https://colab.research.google.com/github/0002F16/hybrid-mobilenetv2-dualconv-eca/blob/main/colab/Train_Eval_TinyImageNet_All20_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/0002F16/hybrid-mobilenetv2-dualconv-eca/blob/main/colab/Train_Eval_TinyImageNet_All20_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Train + evaluate MobileNetV2 variants on Tiny-ImageNet (All 20 runs) (Colab)

This notebook trains and evaluates **4 MobileNetV2 variants** on **Tiny-ImageNet** across 5 seeds (20 total runs):
- `baseline` (MobileNetV2 only)
- `dualconv` (DualConv B4–B10, MobileNetV2 + DualConv)
- `eca` (MobileNetV2 + ECA only)
- `hybrid` (DualConv + ECA, B4–B10)

For each run it uses `experiments/run_train_eval.py` and writes results to:
`OUTPUT_ROOT/tiny_imagenet/<model>/seed_<seed>/metrics.json`.

The Tiny-ImageNet training config comes from `configs/tiny_imagenet.yaml`; this notebook only overwrites `seed` per run (the runner reads `cfg["seed"]`).

You need a **direct-download URL** to a Tiny-ImageNet archive (typically `tiny-imagenet-200.zip`).

Tip: set `MAX_EPOCHS` below for quick sanity runs.

In [ ]:
from pathlib import Path
import os

# If you opened this notebook from a cloned repo, set REPO_DIR accordingly.
# If you need to clone, set GIT_URL to your fork/repo and run the cell.
GIT_URL = "https://github.com/0002F16/hybrid-mobilenetv2-dualconv-eca"
REPO_DIR = Path("/content/hybrid-mobilenetv2-dualconv-eca")

if not REPO_DIR.exists():
    if not GIT_URL:
        raise ValueError("Set GIT_URL to your repo URL, then re-run this cell.")
    os.system(f"git clone {GIT_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)

# Install pinned deps
os.system("pip -q install -r requirements.txt")

# Progress bar for the multi-run loop
os.system("pip -q install tqdm")

In [ ]:
# Paths (feel free to change)
DATA_DIR = Path("/content/data")
OUTPUT_ROOT = Path("/content/outputs")

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Optional epoch cap for fast sanity runs (use None for full Tiny-ImageNet config: 100 epochs).
MAX_EPOCHS = 1

# Tiny-ImageNet direct download URL
TINY_IMAGENET_URL = ""  # required only for Tiny-ImageNet download

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("MAX_EPOCHS:", MAX_EPOCHS)
print("TINY_IMAGENET_URL set:", bool(TINY_IMAGENET_URL))

In [ ]:
# Optional: download + extract Tiny-ImageNet into DATA_DIR
# This ensures the folder layout matches data.preprocessing.get_tiny_imagenet_loaders.

from data.download_tiny_imagenet import ensure_tiny_imagenet

TINY_ROOT = None

if TINY_IMAGENET_URL:
    TINY_ROOT = ensure_tiny_imagenet(url=TINY_IMAGENET_URL, data_dir=DATA_DIR)
    print("Tiny-ImageNet root:", TINY_ROOT)
else:
    # Try common extracted locations under DATA_DIR/tiny_imagenet/.
    candidates = [
        DATA_DIR / "tiny_imagenet" / "tiny-imagenet-200",
        DATA_DIR / "tiny_imagenet" / "tiny_imagenet_200",
        DATA_DIR / "tiny_imagenet" / "Tiny-ImageNet-200",
        DATA_DIR / "tiny_imagenet",
    ]
    for c in candidates:
        if (c / "train").exists() and (c / "val").exists():
            TINY_ROOT = c
            break

if TINY_ROOT is None:
    raise RuntimeError(
        "Tiny-ImageNet root not found. Set TINY_IMAGENET_URL or ensure extracted data exists under /content/data."
    )

print("Using TINY_ROOT:", TINY_ROOT)

In [ ]:
import subprocess
import sys
from copy import deepcopy

import yaml
from tqdm.auto import tqdm

seeds = [42]
#, 123, 3407, 2024, 777]
model_variants = ["baseline"]
#, "dualconv", "eca", "hybrid"]

# Controls for re-running long experiment grids.
SKIP_DONE = True
RESUME_TRAINING = True

# Generate per-seed YAML configs by copying configs/tiny_imagenet.yaml and overwriting `seed`.
base_cfg_path = Path("configs/tiny_imagenet.yaml")
base_cfg = yaml.safe_load(base_cfg_path.read_text())

GEN_CFG_DIR = Path("/content/tiny_imagenet_seed_configs")
GEN_CFG_DIR.mkdir(parents=True, exist_ok=True)

def run(cfg_path: str, *, model: str, dataset_root: Path | None = None, resume: bool = False):
    cmd = [
        sys.executable,
        "experiments/run_train_eval.py",
        "--config",
        cfg_path,
        "--output_root",
        str(OUTPUT_ROOT),
        "--model",
        model,
        "--dataset_root",
        str(dataset_root or DATA_DIR),
    ]
    if MAX_EPOCHS is not None:
        cmd += ["--max_epochs", str(int(MAX_EPOCHS))]
    if resume:
        cmd += ["--resume"]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

def run_dir_for(seed: int, model: str) -> Path:
    # Matches `_run_dir()` in experiments/run_train_eval.py
    return OUTPUT_ROOT / "tiny_imagenet" / str(model).lower() / f"seed_{int(seed)}"

# Pre-generate all per-seed YAML configs once (cheap, keeps configs self-contained).
cfg_paths = {}
for seed in seeds:
    cfg_seed = deepcopy(base_cfg)
    cfg_seed["seed"] = int(seed)
    cfg_out_path = GEN_CFG_DIR / f"tiny_imagenet_seed_{seed}.yaml"
    cfg_out_path.write_text(yaml.safe_dump(cfg_seed, sort_keys=False), encoding="utf-8")
    cfg_paths[int(seed)] = cfg_out_path

run_specs = [(seed, model) for seed in seeds for model in model_variants]

for seed, model in tqdm(run_specs, desc="Tiny-ImageNet (20 runs) "):
    seed_int = int(seed)
    model_lower = str(model).lower()
    run_dir = run_dir_for(seed_int, model_lower)
    metrics_path = run_dir / "metrics.json"
    if SKIP_DONE and metrics_path.exists():
        continue

    resume = RESUME_TRAINING and (run_dir / "checkpoints" / "last.pt").exists()
    run(str(cfg_paths[seed_int]), model=model_lower, dataset_root=TINY_ROOT, resume=resume)

In [ ]:
# Summarize saved metrics (only Tiny-ImageNet runs created by this notebook's output layout)
import json

metrics_paths = sorted(OUTPUT_ROOT.glob("tiny_imagenet/*/seed_*/metrics.json"))
if not metrics_paths:
    print("No metrics.json files found under OUTPUT_ROOT/tiny_imagenet. Run the training cell first.")
else:
    for metrics_path in metrics_paths:
        data = json.loads(metrics_path.read_text())
        ds = data.get("dataset")
        model = data.get("model")
        seed = data.get("seed")
        test_top1 = data.get("test", {}).get("top1_pp")
        test_top5 = data.get("test", {}).get("top5_pp")
        best_epoch = data.get("best_val", {}).get("epoch")
        top5_str = f"{test_top5:.2f}%" if test_top5 is not None else "NA"
        print(f"{ds:12s} | {model:8s} | seed={seed} | best_epoch={best_epoch} | test_top1={test_top1:.2f}% | test_top5={top5_str} | {metrics_path}")